In [3]:
import json
import time
import numpy as np
from sentence_transformers import SentenceTransformer, util
from keybert import KeyBERT

INPUT_FILE = "dataset_sample_10k.json"
OUTPUT_FILE = "coverage_results_rich.json"

# the top 5 concepts
TOP_K = 5

# analyze over a set of thresholds
THRESHOLDS = [0.6, 0.7, 0.8]

# the embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# setting up KeyBERT
kw_extractor = KeyBERT(model=model)

# HELPER FUNCTIONS

def normalize(text):
    return text.lower().strip()


def normalize_keywords(keywords):
    if isinstance(keywords, list):
        return [normalize(k) for k in keywords if isinstance(k, str)]
    elif isinstance(keywords, str):
        return [normalize(keywords)]
    return []


def extract_keywords(text):

    if not text or not text.strip():
        return []

    kws = kw_extractor.extract_keywords(
        text,
        keyphrase_ngram_range=(1, 2),
        stop_words="english",
        top_n=TOP_K
    )

    return [normalize(k[0]) for k in kws]


# MAIN

if __name__ == "__main__":

    start_time = time.time()

    print("🔄 Loading dataset...\n")

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"Loaded {len(data)} datasets\n")

    results = []

    for i, d in enumerate(data, 1):

        # Progress indicator
        if i % 100 == 0:
            print(f"Processed {i}/{len(data)}")

        title = d.get("title", "")
        description = d.get("description", "")

        keywords = normalize_keywords(
            d.get("keywords", [])
        )

        # ------------------------------------------------
        # Encode keywords ONCE per dataset
        # ------------------------------------------------

        keyword_embs = None

        if len(keywords) > 0:
            keyword_embs = model.encode(
                keywords,
                convert_to_tensor=True
            )

        # Extract metadata concepts

        extracted_title = extract_keywords(title)
        extracted_description = extract_keywords(description)

        # Default values
        
        title_coverage_dict = {
            f"title_coverage_{t}": 0.0
            for t in THRESHOLDS
        }

        description_coverage_dict = {
            f"description_coverage_{t}": 0.0
            for t in THRESHOLDS
        }

        avg_title_max_similarity = 0.0
        avg_description_max_similarity = 0.0

        # TITLE COVERAGE

        if len(extracted_title) > 0 and keyword_embs is not None:

            extracted_title_embs = model.encode(
                extracted_title,
                convert_to_tensor=True
            )

            sims_title = util.cos_sim(
                extracted_title_embs,
                keyword_embs
            )

            for t in THRESHOLDS:

                matches = (
                    sims_title >= t
                ).any(dim=1)

                coverage_t = float(
                    matches.sum().item()
                    / len(extracted_title)
                )

                title_coverage_dict[
                    f"title_coverage_{t}"
                ] = coverage_t

            max_sims_title = (
                sims_title.max(dim=1).values
            )

            avg_title_max_similarity = float(
                max_sims_title.mean().item()
            )

        # DESCRIPTION COVERAGE

        if (
            len(extracted_description) > 0
            and keyword_embs is not None
        ):

            extracted_description_embs = model.encode(
                extracted_description,
                convert_to_tensor=True
            )

            sims_description = util.cos_sim(
                extracted_description_embs,
                keyword_embs
            )

            for t in THRESHOLDS:

                matches = (
                    sims_description >= t
                ).any(dim=1)

                coverage_t = float(
                    matches.sum().item()
                    / len(extracted_description)
                )

                description_coverage_dict[
                    f"description_coverage_{t}"
                ] = coverage_t

            max_sims_description = (
                sims_description.max(dim=1).values
            )

            avg_description_max_similarity = float(
                max_sims_description.mean().item()
            )

        # SAVE DATASET RESULT
        
        results.append({

            "dataset_id":
                d.get("dataset_id"),

            "num_keywords":
                len(keywords),

            "num_title_concepts":
                len(extracted_title),

            "num_description_concepts":
                len(extracted_description),

            "avg_title_max_similarity":
                avg_title_max_similarity,

            "avg_description_max_similarity":
                avg_description_max_similarity,

            **title_coverage_dict,
            **description_coverage_dict
        })

    # ROBUSTNESS SUMMARY
    
    robustness_summary = {}

    for t in THRESHOLDS:

        robustness_summary[
            f"title_coverage_mean_{t}"
        ] = float(
            np.mean([
                r[f"title_coverage_{t}"]
                for r in results
            ])
        )

        robustness_summary[
            f"description_coverage_mean_{t}"
        ] = float(
            np.mean([
                r[f"description_coverage_{t}"]
                for r in results
            ])
        )

    # SAVE RESULTS

    output_payload = {
        "dataset_results": results,
        "robustness_summary": robustness_summary
    }

    with open(
        OUTPUT_FILE,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            output_payload,
            f,
            indent=2
        )

    end_time = time.time()

    print(
        f"\n💾 Results saved to {OUTPUT_FILE}"
    )

    print(
        f"\n⏱ Total time: "
        f"{end_time - start_time:.2f}s"
    )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

🔄 Loading dataset...

Loaded 10000 datasets

Processed 100/10000
Processed 200/10000
Processed 300/10000
Processed 400/10000
Processed 500/10000
Processed 600/10000
Processed 700/10000
Processed 800/10000
Processed 900/10000
Processed 1000/10000
Processed 1100/10000
Processed 1200/10000
Processed 1300/10000
Processed 1400/10000
Processed 1500/10000
Processed 1600/10000
Processed 1700/10000
Processed 1800/10000
Processed 1900/10000
Processed 2000/10000
Processed 2100/10000
Processed 2200/10000
Processed 2300/10000
Processed 2400/10000
Processed 2500/10000
Processed 2600/10000
Processed 2700/10000
Processed 2800/10000
Processed 2900/10000
Processed 3000/10000
Processed 3100/10000
Processed 3200/10000
Processed 3300/10000
Processed 3400/10000
Processed 3500/10000
Processed 3600/10000
Processed 3700/10000
Processed 3800/10000
Processed 3900/10000
Processed 4000/10000
Processed 4100/10000
Processed 4200/10000
Processed 4300/10000
Processed 4400/10000
Processed 4500/10000
Processed 4600/1000